# Mamba SOH — LFP chemistry variant (GH-67 Mức 2)

Retrain kiến trúc `MambaSOHPredictor` (window=30, d_model=64, d_state=16) trên dataset
**Severson et al. 2019** (Nature Energy) — LFP/graphite A123 APR18650M1A, 1.1 Ah — để có bộ
artifact riêng cho chemistry LFP. **KHÔNG đụng** model NASA/NMC production (`soh_mamba_v1.6.pth`).

---

## Checklist trước khi Run All

| # | Việc | Ghi chú |
|---|------|---------|
| 1 | Settings → Accelerator → **GPU T4 x2** | KHÔNG chọn P100 — PyTorch Kaggle đã bỏ sm_60 |
| 2 | **+ Add Data** → dataset chứa `.mat` Severson | Batch1/2/3(/4) từ https://data.matr.io/1/ |
| 3 | Add-ons → Secrets → `GITHUB_TOKEN` | GitHub PAT (nếu repo private) |
| 4 | **`git push` code mới lên GitHub TRƯỚC** | Cell 3 sẽ kiểm tra và dừng nếu thiếu |

> ⚠️ **Mục 4 là chỗ dễ mất thời gian nhất.** Notebook clone code từ GitHub remote — sửa file
> trên máy local mà chưa push thì Kaggle **không thấy**. Lần chạy 2026-07-25 mất ~11 giờ và ra
> kết quả y hệt lần trước vì lý do này. Cell 3 giờ tự chặn trường hợp đó.

## Lịch sử kết quả

| Lần | Cấu hình | Test MAE | Test RMSE | Đạt target |
|-----|----------|----------|-----------|------------|
| 1 | `--epochs 5`, `CYCLE_COUNT_NORM=200` | 2.5856% | 3.4745% | ❌ |
| 2 | `--epochs 5`, `CYCLE_COUNT_NORM=2300` | 1.9213% | 2.7627% | ✅ (che lỗi EOL) |
| 3 | + pha xả + lọc outlier + `--balance-bands`, 50 epoch | **1.4365%** | **1.8767%** | ✅ |
| 4 | + `time` giây (#7) + chặn nhiễu feature (#8) + `--swa` | 1.3095% | 1.9228% | ✅ |
| 5 | + sàn nhiễu 1e-7, hiện dải SOH≥100 | **1.2899%** | **1.8935%** | ✅ |
| 6 | + clip SOH 100 + `--jitter 0.01` + stride 3 | 2.8390% | 3.5785% | ❌ **lùi** |
| 7 | + lọc đoạn xả dài (#9) + `soc_percent` đúng nghĩa (#10) | ? | ? | — |

Target chính thức: **MAE < 2.0%** · **RMSE < 3.0%** — đã đạt từ lần 3.
Tham chiếu NASA v1.6 cùng kiến trúc: 1.34% / 1.84%. **Mục tiêu mới: cả hai < 1.0%.**

> RMSE mới là ràng buộc thật: RMSE ≥ MAE luôn, tỉ lệ hiện tại 1.31 → muốn RMSE < 1.0 thì MAE
> phải xuống ~0.77%, tức cải thiện **47%**. Chưa model nào của dự án (kể cả NASA) xuống dưới 1%.

**Vì sao kỳ vọng nhiều ở fix #7:** mọi window cắt từ cùng 1 chu kỳ xả mang **chung 1 nhãn SOH**.
Lát đầu xả (3.4 V) và lát cuối xả (2.9 V) trông khác hẳn nhau nhưng phải cho cùng con số. Model
chỉ phân biệt được nếu biết **đang ở đoạn nào của quá trình xả** — tín hiệu đó chính là
`soc_percent`, mà nó đang chết vì bug #7. Fix này tấn công trực diện nguồn sai số lớn nhất.

## 🚨 Bug #7 — cột `time` của Severson là PHÚT, không phải giây

Đo trên artifact lần 3: scaler fit `time` trên **[0.000, 24.130]**. Xả 4C của cell 1.1 Ah kéo
dài ~15 phút → 24.13 là **phút**. NASA và payload BE gửi đều dùng **giây**. Hai hậu quả:

1. **Kênh `soc_percent` chết lúc train** — `compute_soc_percent()` làm `time / 3600` (giả định
   giây). Cho ăn phút → đếm thiếu 60×: window 12 phút cho `soc 100.00 → 98.79`, đúng phải là
   `100.00 → 27.27`. Model mất 1 trong 6 kênh input.
2. **Lệch phân bố lúc inference** — BE gửi giây, scaler fit trên phút: `60 s → 2.49`,
   `300 s → 12.43`, `720 s → 29.84`, trong khi train chỉ thấy `[0, 1]`. Không lỗi nào được raise.

Đã fix bằng `--time-unit minutes` (mặc định) quy về giây ngay lúc parse.

## 🚨 Bug #8 — 7 đặc trưng đang khuếch đại nhiễu làm tròn 15.000–93.000×

Đo trên `feature_scaler_lfp.pkl` lần 3: 7/57 đặc trưng có `var` ở mức **1e-10 … 4e-9**, tức chỉ
là **nhiễu làm tròn số thực**. `StandardScaler` chia cho `sqrt(var)` → biến nhiễu đó thành tín
hiệu biên độ đơn vị:

| Đặc trưng | var | Khuếch đại |
|-----------|-----|------------|
| `spec.temp.centroid` | 1.15e-10 | **93.072×** |
| `spec.temp.band_mid` | 1.70e-10 | 76.739× |
| `spec.temp.band_high` | 2.54e-10 | 62.733× |
| `spec.temp.band_low` | 8.26e-10 | 34.788× |
| `spec.temp.gini` | 1.40e-09 | 26.727× |
| `stat.temp.waveform` | 2.60e-09 | 19.610× |
| `spec.temp.flatness` | 4.32e-09 | 15.210× |

Gần như trọn khối **phổ nhiệt độ**. Severson chạy trong buồng 30 °C nên nhiệt độ gần như không
đổi trong window 30 bước → phổ FFT là DC thuần → mọi mô tả hình dạng phổ suy biến.

7 kênh nhiễu này vào thẳng `film_proj` — mạng sinh `gamma`/`beta` **điều biến MỌI hidden unit**
— nên nhiễu lan ra toàn bộ biểu diễn và biểu hiện thành phương sai dự đoán, tức **RMSE**.

Fix: `FEATURE_VAR_FLOOR = 1e-7`, ép `scale_ = 1.0` cho các đặc trưng dưới ngưỡng → giá trị co về
~0 (input chết) thay vì input nhiễu. (Ban đầu đặt `1e-8` nhưng `spec.temp.entropy` với
`var = 2.89e-8` lọt lưới, vẫn khuếch đại 5.879×.) Ngưỡng `1e-7` bắt 13 đặc trưng (var ≤ 2.89e-8)
và giữ nguyên 5 đặc trưng có tín hiệu thật (var ≥ 3.84e-7) — cách nhau **13,3×**.
Lần 5 xác nhận: khuếch đại lớn nhất còn lại **1.615×**.
**Không phải sửa code inference** — `inference.py` gọi `feature_scaler.transform()` trên chính
file pickle này nên tự động khớp.

Output: `soh_mamba_v2.0-lfp.pth`, `isolation_forest_v2.0-lfp.pkl`, `scaler_lfp.pkl`,
`feature_scaler_lfp.pkl` — 4 file riêng, không ghi đè artifact NASA.

## 1 — GPU check

In [ ]:
!nvidia-smi
import torch
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Chua bat GPU: Settings -> Accelerator -> GPU T4 x2'
name = torch.cuda.get_device_name(0)
print('GPU:', name)
assert 'P100' not in name, 'P100 khong tuong thich PyTorch Kaggle (sm_60) - doi sang GPU T4 x2'

## 2 — Clone repo

In [ ]:
import subprocess, os
BRANCH = 'feat/GH-67-lfp-retrain-severson'
REPO   = '/kaggle/working/ai-module'
url = 'https://github.com/GSU26SE55/ai-module.git'
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    url = f'https://{token}@github.com/GSU26SE55/ai-module.git'
except Exception as e:
    print('Khong co GITHUB_TOKEN secret -> thu public clone:', e)

if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, url, REPO], check=True)
os.chdir(REPO)
print(subprocess.check_output(['git', 'log', '-1', '--format=%h %ci %s']).decode())

## 3 — Kiểm tra code clone về có đúng bản mới không

Chặn đúng cái bẫy đã làm mất ~11 giờ: notebook đã sửa nhưng code trên GitHub thì chưa.
Nếu cell này fail → về máy chạy `git push` rồi **Restart & Run All** (phải xoá session để
clone lại, vì cell 2 skip clone khi thư mục đã tồn tại).

In [ ]:
import pathlib

checks = {
    'scripts/preprocess_lfp.py': ['--cycle-stride', 'PHYSICAL_RANGES', '_nonphysical_channel',
                                  '--phase', '_longest_discharge_segment',
                                  'TIME_UNIT_SECONDS', '--time-unit', 'FEATURE_VAR_FLOOR',
                                  '--soh-clip', 'SOH_CLIP_DEFAULT', 'MAX_DISCHARGE_SECONDS',
                                  '--soc-mode'],
    'scripts/train.py':          ['--feature-scaler-version', '--mamba-out', '--iso-out'],
    'src/core/config.py':        ['LFP_CYCLE_COUNT_NORM', 'LFP_NOMINAL_CAPACITY_AH'],
}
missing = []
for path, needles in checks.items():
    text = pathlib.Path(path).read_text(encoding='utf-8')
    for n in needles:
        status = 'OK  ' if n in text else 'THIEU'
        print(f'  [{status}] {path}: {n}')
        if n not in text:
            missing.append(f'{path}::{n}')

assert not missing, (
    'Code clone ve THIEU cac fix sau: ' + ', '.join(missing) +
    '\n-> Ve may chay: git add -A && git commit && git push, roi Restart & Run All.'
)
print('\nTat ca fix da co trong code clone ve.')

## 4 — Dependencies

In [ ]:
%pip install -q h5py scipy scikit-learn joblib pandas
import h5py, scipy, sklearn
print('h5py', h5py.__version__, '| scipy', scipy.__version__, '| sklearn', sklearn.__version__)

## 5 — Tìm dataset Severson

Cần ít nhất 1 file `*.mat` có chữ "batch" trong tên (vd
`2017-05-12_batchdata_updated_struct_errorcorrect.mat`). Fail → kiểm tra lại **+ Add Data**.

In [ ]:
import os, subprocess
found = [f for f in subprocess.check_output(
    ['find', '/kaggle/input', '-iname', '*batch*.mat']).decode().splitlines() if f]
assert found, 'Khong thay file *batch*.mat - dung + Add Data de attach dataset Severson truoc'
DATASET = os.path.dirname(found[0])
os.chdir('/kaggle/working/ai-module')
print('DATASET:', DATASET)
for f in found:
    print('  ', f)

## 6 — Preprocess (Severson `.mat` → window=30, 6 feature)

Hai fix **đúng đắn** (không phải tuning) áp dụng ở bước này, mặc định đã bật:

- **`--phase discharge`** — chỉ lấy đoạn **xả** dài nhất của mỗi cycle. Severson lưu nguyên
  cycle gồm cả sạc nhanh nhiều bước (dòng dương tới ~8 A) lẫn xả 4C, trong khi NASA
  (`scripts/preprocess.py`) chỉ nạp chu kỳ xả, và inference cũng chỉ thấy telemetry xả.
  Trước fix này ~nửa số window là pha **sạc** — model phải đoán nhãn dung lượng *xả*
  (`QDischarge`) từ mẫu sạc.
- **`PHYSICAL_RANGES`** — drop cycle chứa giá trị sensor phi vật lý.

### Ngân sách thời gian — đo thật, không đoán

| Cấu hình | Window train | Thời gian |
|----------|--------------|-----------|
| Full data, 5 epoch (lần 2) | 3 220 853 | 11,2 h (sát giới hạn 12h) |
| stride 5 + pha xả, 50 epoch (lần 5) | 230 644 | **3,9 h** (23 phút preprocess + 3,6 h train) |
| **stride 3 + pha xả, 50 epoch (lần này)** | ~384 000 | **~6,4 h** ước tính |

Lần 5 chỉ dùng 3,9/12 giờ nên còn nhiều dư địa. Hạ stride 5 → 3 cho **thêm 66% dữ liệu** —
đây cũng là cách chống overfit hiệu quả nhất (xem §7).

### 🚨 Bug #10 — `soc_percent` vô dụng khi train, lệch hẳn lúc inference

Đây là phát hiện cấu trúc lớn nhất, không phải lỗi outlier như #6/#8/#9.

`compute_soc_percent()` được thiết kế **window-local** — docstring ghi rõ *"SOC is defined
RELATIVE TO THE WINDOW: 100% at the first row"*. Preprocess gọi nó trên từng lát 30 dòng, nên:

```
Train  (window-local): MOI window deu 100.0 -> 91.2%   range [0.912, 1.000]  bien thien 8.8 diem
Inference (BE 6 cot):  SOC thuc cua pin                 range [0.094, 1.000]  bien thien 90.6 diem
```

Ba hậu quả:

1. **Một kênh input bị bỏ không.** `soc_percent` gần như hằng số → model thực chất chỉ có **5
   kênh hữu ích, không phải 6**.
2. **Mất đúng tín hiệu quan trọng nhất.** Mọi window của một chu kỳ mang **chung 1 nhãn SOH**,
   nên model chỉ phân biệt lát đầu (3.4 V) với lát cuối (2.9 V) qua tín hiệu **vị trí**. Tính
   trên toàn đoạn xả thì `soc` biến thiên 90,6 điểm; window-local thì ~0.
3. **Lệch phân bố train/inference.** Model chưa bao giờ thấy `soc_norm < 0.912` khi train, nhưng
   BE gửi xuống tận `0.094` — mà payload 6 cột là **default BE dùng thật**.

Fix: `--soc-mode cycle` (mặc định) — Coulomb-count trên **toàn đoạn xả** rồi lát ra theo window.
Verify: `cycle` mode biến thiên 90,6 điểm và khớp đúng dải BE gửi; `--soc-mode window` giữ lại
để ablation.

### 🚨 Bug #9 — đoạn xả dài bất thường làm nổ dải `time` (nguyên nhân lần 6 lùi)

Lần 6 đổi `--cycle-stride 5 → 3` và **tệ đi hẳn**: MAE 1.2899% → **2.8390%**, RMSE 1.8935% →
**3.5785%** (vượt cả target 3%). Nguyên nhân nằm trong metadata scaler:

```
Lần 5:  time [0.000,  1,447.789]   <- ~24 phút, hợp lý cho xả 4C
Lần 6:  time [0.000, 25,551.094]   <- 7,1 GIỜ
```

Dải rộng ra **17,6×**. Stride 3 lấy tập cycle khác stride 5 và vô tình lấy được một cycle có đoạn
"xả" dài 7,1 giờ. Hệ quả: window xả bình thường ~900 s từ chỗ chiếm 62% dải `[0,1]` tụt còn
**3,5%** — kênh `time` mất 17,6× độ phân giải.

`time` chính là tín hiệu **vị-trí-trong-chu-kỳ-xả** — mọi window của một chu kỳ mang chung 1 nhãn
SOH nên model chỉ phân biệt lát đầu/lát cuối qua `time`/`soc_percent`. Bóp nó 17,6× thì sai số
tăng gấp đôi là hợp lý.

Cùng loại lỗi với #6 (temperature `[-270, 400]`) và #8 (nhiễu feature): **một mẫu vô lý làm hỏng
một dải và bóp chết tín hiệu thật.**

Fix: `MAX_DISCHARGE_SECONDS = 7200` (2 h — gấp 8 lần xả 4C nominal ~900 s). Loại đoạn xả dài hơn
thế, kèm in `p50/p95/p99/max` của thời lượng và **cảnh báo lớn** nếu dải `time` vẫn vượt ngưỡng.

### `--soh-clip 100`

Lần 5 làm lộ ra dải trước đây vô hình: `SOH 100-110: n=532 MAE=3.533% bias=-3.506%` — chỉ 3,2%
số mẫu nhưng gánh 8,7% tổng sai số tuyệt đối. Đó là các chu kỳ đầu đời có dung lượng đo **vượt**
nominal 1.1 Ah → nhãn SOH > 100%.

SOH nghĩa là "sức khoẻ so với pin mới", nên >100% không phải một trạng thái sức khoẻ mà là hệ quả
của việc chia cho nominal ghi trên datasheet. Không có gì phía sau phân biệt 100% với 103%:
ngưỡng `health_stage` là 80/85/90 nên cả hai đều "Healthy". Clip (thay vì loại bỏ) giữ lại window
làm tín hiệu train cho vùng "rất khoẻ" mà bỏ đi một mục tiêu lệch vô ích.

> ⚠️ **Trung thực khi báo cáo:** clip cũng đổi nhãn của **tập test**, nên một phần cải thiện là do
> định nghĩa chứ không phải model giỏi lên. Phải nói rõ điều này khi so với lần 5 (1.2899%/1.8935%).

### Đọc 4 thứ trong log

1. **`scaler range time`** — lần 3 ra `[0.000, 24.130]` (phút). Lần này phải ra khoảng
   **`[0.000, ~1450]`** (giây). Nếu vẫn ~24 → fix `--time-unit` chưa ăn.
2. **`scaler range temperature`** — kỳ vọng ~`[0, 45]`, không còn `[-270, 400]`.
3. **`scaler range current`** — phải **toàn số âm** (~`[-4.3, -0.1]`) = thuần pha xả.
4. **`discharge duration trung vi`** — sau fix phải in `giay (khop NASA)`, không phải `PHUT?`.
5. **`N/57 feature suy bien ... -> ep scale_=1.0`** — kỳ vọng ~7–12 feature (bug #8). Nếu in ra
   `0` thì fix `FEATURE_VAR_FLOOR` chưa ăn.
6. **Số cycle bị drop** — vài chục là bình thường. Drop tỉ lệ lớn → đang vứt dữ liệu thật.

Ngoài ra `[TIMING]` tách thời gian parse `.mat` vs trích xuất window/feature + số step/epoch.

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/preprocess_lfp.py --data-dir "{DATASET}" --output-dir data/processed_lfp \
    --cycle-stride 3 --phase discharge --time-unit minutes     --soh-clip 100 --soc-mode cycle

## 7 — Train

Kiến trúc + hyperparameter **giữ nguyên** như model NASA. `--mamba-out`/`--iso-out`/
`--model-version` đảm bảo KHÔNG ghi đè `soh_mamba_v1.6.pth` production.

### ⚠️ Lần 5 cho thấy model đang OVERFIT — đây là lý do có `--jitter`

```
epoch 30: train 0.000348 | val 0.000505   <- val TỐT NHẤT
epoch 40: train 0.000326 | val 0.000589   <- train giảm, val TĂNG
epoch 50: train 0.000310 | val 0.000549
```

Train loss giảm đều nhưng val loss chạm đáy ở epoch 30 rồi đi lên → overfit rõ. Cũng giải thích
vì sao SWA bị revert **2 lần liên tiếp**: trung bình hoá các epoch cuối vốn đã overfit thì không
giúp được gì.

- **`--jitter 0.01` (mới)** — cộng nhiễu Gauss vào input mỗi bước train. Chống overfit trực tiếp.
  Nếu val loss vẫn tách khỏi train loss thì tăng lên `0.02`; nếu **cả hai** cùng cao hơn lần 5
  thì hạ xuống `0.005` (nhiễu quá mạnh gây underfit).
- **`--cycle-stride 3`** thay vì 5 — thêm 66% dữ liệu, cách chống overfit hiệu quả nhất.
- **`--balance-bands`** (giữ) — trọng số loss tỉ lệ nghịch tần suất ô (nhiệt độ × SOH).
- **`--swa`** (giữ) — rủi ro bằng 0 vì `train.py` tự revert nếu val_loss tệ hơn. Đã bị revert 2
  lần; nếu lần này `--jitter` khắc phục được overfit thì SWA có thể bắt đầu có tác dụng.

`--epochs 50`: với ~290k window, 50 epoch cho `ReduceLROnPlateau` ~10 lần cơ hội giảm LR và
early-stopping (`patience=15`) có chỗ hoạt động — điều chưa từng xảy ra ở lần 1/2 (5 epoch).

`train.py` chỉ in dòng metric khi `epoch % 10 == 0` → sẽ có 5 dòng. So `TrainLoss` với `ValLoss`:

- cả 2 còn cao và đang giảm → vẫn underfit, tăng epoch
- TrainLoss thấp mà ValLoss cao → overfit, thêm `--jitter 0.01`
- cả 2 phẳng sớm → đã hội tụ

Tìm thêm dòng `SWA: averaged N epochs | SWA val_loss=... vs best-ckpt val_loss=...` để biết SWA
có được dùng hay bị revert.

### Còn lại cho lần sau nếu vẫn chưa dưới 1%

`train()` cho window=30 chỉ nhận 4 knob: `epochs`, `--balance-bands`, `--jitter`, `--swa`
(mọi flag khác như `--pooling`/`--patch-size`/`--dropout` **chỉ dùng cho `--long`**, vô tác dụng
ở đây). Lần này dùng hết 3 knob an toàn. Còn lại:

1. `--cycle-stride 2` — gấp 2.5 lần dữ liệu, phải giảm còn ~25 epoch để vừa 12h
2. `--jitter 0.01` — nếu log cho thấy overfit
3. Tăng `d_model` 64→128 — **lệch spec `CLAUDE.md`**, cần quyết định riêng

In [ ]:
import os; os.chdir('/kaggle/working/ai-module')
!python scripts/train.py \
    --data-dir data/processed_lfp \
    --epochs 50 \
    --balance-bands \
    --swa \
    --jitter 0.01 \
    --log-dir logs/training \
    --mamba-out models/weights/soh_mamba_v2.0-lfp.pth \
    --iso-out models/weights/isolation_forest_v2.0-lfp.pkl \
    --model-version 2.0-lfp \
    --feature-scaler-version 2.0-lfp

## 8 — Đóng gói artifact để tải về

**Chạy TRƯỚC bước chẩn đoán có chủ đích.** Ở lần 4 một `assert` chẩn đoán đã fail và giết cả
notebook **sau khi train xong 4 tiếng**, khiến bước đóng gói không chạy. Giờ artifact được gom
vào zip ngay khi train xong, trước bất cứ kiểm tra nào.

4 file cần copy vào `models/weights/` trên máy. **Không tự commit** — tải zip về, tự
`git add` + `git commit` + `git push` theo quy trình repo.

In [ ]:
import shutil, os
OUT = '/kaggle/working/lfp_artifacts'
os.makedirs(OUT, exist_ok=True)
for p in [
    'models/weights/soh_mamba_v2.0-lfp.pth',
    'models/weights/isolation_forest_v2.0-lfp.pkl',
    'models/weights/scaler_lfp.pkl',
    'models/weights/feature_scaler_lfp.pkl',
]:
    shutil.copy(p, OUT)
    print('  +', os.path.basename(p), f'({os.path.getsize(p)/1024:.0f} KB)')
shutil.make_archive(OUT, 'zip', OUT)
print('\nTai ve: lfp_artifacts.zip (tab Output ben phai)')

## 9 — Kiểm tra kết quả

**Nhìn `Per-band` trước, MAE tổng sau.** MAE tổng bị dải đông mẫu chi phối nên luôn trông đẹp.
Chỉ số quyết định là **bias ở dải 70–80%**: lần 2 là +10.280% (pin 75% bị báo thành 85% → bỏ
sót pin cần thay), lần 4 đã xuống **+2.474%**.

Cell này **chỉ cảnh báo, không `assert`** — chẩn đoán không được phép giết một lần chạy nhiều
giờ. Thấy dòng `[!]` nào thì đọc kỹ, nhưng artifact ở cell 8 vẫn đã an toàn.

In [ ]:
import glob, os, re
import torch, joblib

PREV = {'mae': 1.2899, 'rmse': 1.8935, 'bias_70_80': +2.560}   # lan 5 = TOT NHAT (lan 6 lui)
def chk(ok, msg):
    print(('  [OK] ' if ok else '  [!]  ') + msg)   # canh bao, KHONG assert
ck = torch.load('models/weights/soh_mamba_v2.0-lfp.pth', map_location='cpu', weights_only=False)
mae, rmse = ck['test_mae'], ck['test_rmse']

print('=== PER-BAND (quan trong nhat) ===')
logs = sorted(glob.glob('logs/training/train_*.log'), key=os.path.getmtime)
band_lines = []
if logs:
    for ln in open(logs[-1], encoding='utf-8'):
        if re.search(r'SOH\s+\d+-\d+', ln):
            band_lines.append(ln.rstrip())
            print('  ', ln.split('INFO')[-1].strip())
if not band_lines:
    print('   (khong doc duoc log - xem truc tiep output cell 7)')
else:
    # Tim dung DONG cua dai 70-80 roi moi bat bias — join cac dong lai va dung
    # `.*` se tham lam, vot nham bias cua dong cuoi cung.
    hit = next((l for l in band_lines if re.search(r'SOH\s*70-80', l)), None)
    m = re.search(r'bias=([+-][\d.]+)', hit) if hit else None
    if m:
        now = float(m.group(1))
        print(f"\n   bias dai 70-80%: {now:+.3f}%  (lan 5: {PREV['bias_70_80']:+.3f}%)"
              f"  -> thay doi {abs(PREV['bias_70_80']) - abs(now):+.3f} diem")
    else:
        print('\n   (dai 70-80% khong co mau trong test set lan nay)')

print('\n=== METRIC TONG ===')
print(f"MAE : {mae:.4f}%  (target <2.0, TOT NHAT {PREV['mae']})  -> {PREV['mae'] - mae:+.4f}")
print(f"RMSE: {rmse:.4f}%  (target <3.0, TOT NHAT {PREV['rmse']})  -> {PREV['rmse'] - rmse:+.4f}")
print('Dat target:', mae < 2.0 and rmse < 3.0)
print('balance_bands:', ck.get('balance_bands'), '| jitter:', ck.get('jitter'), '| swa:', ck.get('swa'))

print('\n=== XAC NHAN FIX PREPROCESS DA AN ===')
s = joblib.load('models/weights/scaler_lfp.pkl'); sc = s['scaler']
for i, n in enumerate(['voltage', 'current', 'temperature', 'time']):
    print(f'  {n:<12}: [{sc.data_min_[i]:10.3f}, {sc.data_max_[i]:10.3f}]')
print('  metadata:', {k: v for k, v in s.items() if k not in ('scaler', 'trained_on')})
fs = joblib.load('models/weights/feature_scaler_lfp.pkl')
amp_max = float((1.0 / fs['scaler'].scale_).max())
print(f"  feature_scaler: var_floor={fs.get('var_floor')} n_degenerate={fs.get('n_degenerate')}")
print(f'  khuech dai lon nhat con lai: {amp_max:,.0f}x  (lan 3: 93,072x | lan 4: 5,879x | lan 5: 1,615x)')
print()
chk(s.get('phase') == 'discharge', "phase=discharge (khong phai code cu)")
chk(sc.data_max_[1] <= 0, "current toan AM (thuan pha xa)")
chk(sc.data_max_[3] > 200, f"time tinh bang GIAY (max={sc.data_max_[3]:.1f}, ky vong ~1450)")
chk(fs.get('n_degenerate') is not None, "feature_scaler co n_degenerate")
chk(amp_max < 2000, f"khong con feature khuech dai nhieu (amp_max={amp_max:,.0f}x)")

## 10 — Dọn output (xoá repo clone — artifact đã nằm trong zip)

In [ ]:
import os, shutil
os.chdir('/kaggle/working')
shutil.rmtree('/kaggle/working/ai-module', ignore_errors=True)
shutil.rmtree('/kaggle/working/lfp_artifacts', ignore_errors=True)
print('Output con lai:', sorted(os.listdir('/kaggle/working')))